In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from src.split import split_breaths, rc_strata
from src.data import load_data

In [3]:
train_raw = load_data('../data/train.csv')
test_raw = load_data('../data/test.csv')

train_part, val_part = split_breaths(train_raw, rc_strata(train_raw))

In [4]:
def breath_table(df: pd.DataFrame) -> pd.DataFrame:
    """One row per breath: u_in at each of 80 steps, duration, n_insp, R_C combo."""
    g = df.groupby("breath_id")
    signal = (
        df.assign(step=g.cumcount())
        .pivot(index="breath_id", columns="step", values="u_in")
        .add_prefix("u_in_")
    )
    summary = pd.DataFrame(
        {
            "duration": g["time_step"].max(),
            "n_insp": (df["u_out"] == 0).groupby(df["breath_id"]).sum(),
            "R_C": (df["R"].astype(str) + "_" + df["C"].astype(str)).groupby(df["breath_id"]).first(),
        }
    )
    return pd.concat([signal, summary], axis=1)


train_b = breath_table(train_part)
val_b = breath_table(val_part)
test_b = breath_table(test_raw)
train_b.shape, val_b.shape, test_b.shape

((60360, 83), (15090, 83), (50300, 83))

In [ ]:
def adversarial_auc(a: pd.DataFrame, b: pd.DataFrame, cv: int = 5) -> str:
    X = pd.get_dummies(pd.concat([a, b]), columns=["R_C"])  
    y = [0] * len(a) + [1] * len(b)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    return f"{scores.mean():.4f} ± {scores.std():.4f}"

In [6]:
print("train vs val :", adversarial_auc(train_b, val_b))
print("train vs test:", adversarial_auc(train_b, test_b))

train vs val : 0.4908 ± 0.0030
train vs test: 0.4992 ± 0.0038


so, there is no data leak between our data